**GroupBy & Aggregations**

| Function | What it does | Example |
|----------|--------------|---------|
| `groupBy(col)` | Groups rows by one or more columns | `df.groupBy("region")` |
| `count()` | Counts the number of rows in each group | Orders per region |
| `sum(col)` | Calculates the sum of a numeric column | Total sales |
| `avg(col)` | Calculates the average of a numeric column | Average order value |
| `mean(col)` | Same as `avg()` | Mean quantity |
| `min(col)` | Returns the minimum value | Lowest price |
| `max(col)` | Returns the maximum value | Highest price |
| `agg()` | Performs multiple aggregations at once | `sum`, `avg`, `count` together |
| `countDistinct(col)` | Counts distinct values | Unique customers |
| `first(col)` | Returns the first value in each group | First order date |
| `last(col)` | Returns the last value in each group | Last order date |
| `collect_list(col)` | Collects values into a list (duplicates kept) | All products ordered |
| `collect_set(col)` | Collects unique values into a set | Unique payment methods |

In [7]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-15")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')


In [2]:
orders_df=spark.read.\
    option("header","true").\
        option("inferSchema","true").\
            csv("s3a://pyspark-30-days-rahul-2026/data/orders.csv")
products_df=spark.read.\
    option("header","true").\
        option("inferSchema","true").\
            csv("s3a://pyspark-30-days-rahul-2026/data/products.csv")
customers_df=spark.read.\
    option("header","true").\
        option("inferSchema","true").\
            csv("s3a://pyspark-30-days-rahul-2026/data/customers.csv")

**Task 1**

From orders.csv, calculate the total revenue, order count, and average order value per status. Sort by total revenue descending.

In [3]:
from pyspark.sql import functions as F

orders_df.withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy(
    "status"
).agg(
    F.round(F.sum("revenue"), 2).alias("total_revenue"),
    F.count("order_id").alias("order_count"),
    F.round(F.avg("revenue"), 2).alias("avg_order_value")
).orderBy(
    F.col("total_revenue").desc()
).show(truncate=False)

+----------+-------------+-----------+---------------+
|status    |total_revenue|order_count|avg_order_value|
+----------+-------------+-----------+---------------+
|Delivered |33233.3      |74         |449.1          |
|Shipped   |5789.75      |12         |482.48         |
|Processing|4159.72      |10         |415.97         |
|Cancelled |1199.91      |4          |299.98         |
+----------+-------------+-----------+---------------+



**Task 2**

From orders.csv, group by region and payment_method. Calculate order count and total revenue. Filter to show only groups with more than 5 orders.

In [4]:
orders_df.withColumn(
    "revenue",
    F.col("quantity")*F.col("unit_price")).\
        groupBy("region","payment_method").\
            agg(
                F.count("order_id").alias("order_count"),
                F.round(F.sum('revenue'),2).alias("total revenue")
                
            ).where(F.col('order_count')>5).show()

+-------+--------------+-----------+-------------+
| region|payment_method|order_count|total revenue|
+-------+--------------+-----------+-------------+
|   West|        PayPal|         16|      5844.72|
|   West|   Credit Card|         16|      6949.63|
|  South|   Credit Card|         12|      5099.79|
|   East|   Credit Card|         24|     11159.39|
|Midwest|   Credit Card|         12|      7519.62|
+-------+--------------+-----------+-------------+



**Task 3**

Join orders.csv with products.csv on product_id. Group by category and calculate total revenue, order count, and average quantity. Which category has the highest revenue?

In [5]:
from pyspark.sql import functions as F

joined_df = (
    orders_df.alias("o")
    .join(
        products_df.alias("p"),
        on="product_id",
        how="inner"
    )
)

joined_df.withColumn(
    "revenue",
    F.col("o.unit_price") * F.col("o.quantity")
).groupBy(
    F.col("p.category")
).agg(
    F.round(F.sum("revenue"), 2).alias("total_revenue"),
    F.count("o.order_id").alias("order_count"),
    F.round(F.avg("o.quantity"), 2).alias("average_quantity")
).orderBy(
    F.col("total_revenue").desc()
).show(truncate=False)

+---------------+-------------+-----------+----------------+
|category       |total_revenue|order_count|average_quantity|
+---------------+-------------+-----------+----------------+
|Electronics    |35088.37     |75         |2.17            |
|Furniture      |9009.39      |22         |2.77            |
|Office Supplies|284.92       |3          |2.67            |
+---------------+-------------+-----------+----------------+



**Task 4**

From orders.csv, find the top 5 customers by total spend. Join with customers.csv to show their full name alongside their total spend and order count.

In [6]:
joined_df2=orders_df.join(customers_df,
on='customer_id',how='inner'
)
joined_df2.withColumn(
    "spends",
    F.col("unit_price")*F.col("quantity")*(1-F.col("discount_pct")/100)
).groupBy('customer_id').\
    agg(F.round(F.sum('spends')).alias("total_spend"),
    F.count("order_id").alias("order_count")).\
    orderBy(F.col("total_spend").desc()).show(5)

+-----------+-----------+-----------+
|customer_id|total_spend|order_count|
+-----------+-----------+-----------+
|       C001|     3450.0|          5|
|       C003|     2688.0|          5|
|       C017|     2190.0|          4|
|       C002|     2076.0|          5|
|       C006|     2040.0|          4|
+-----------+-----------+-----------+
only showing top 5 rows
